# Lektion 12 - Minskning av chatthistorik med Agent Scratchpad

Den här anteckningsboken visar hur man hanterar kontext i långa samtal med Microsoft Agent Framework. När samtal växer ökar antalet token — och överskrider så småningom modellens kontextfönster. Vi löser detta med ett **mönster för kontextsummering** och en **agent scratchpad** för bestående minne.

## Vad du kommer att lära dig:
1. **Varför kontexthantering är viktigt**: Förstå tokenbegränsningar och kontextfönster
2. **Konstextmedvetna agenter**: Bygga agenter som hanterar sin egen samtalskontext
3. **Mönster för kontextsummering**: Använda verktyg för att kondensera samtalshistorik
4. **Agent Scratchpad**: Bestående minne som överlever kontextminskning

## Förkunskaper:
- Azure OpenAI-konfiguration med miljövariabler inställda
- Grundläggande förståelse för agentkoncept från tidigare lektioner


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Varför hantering av kontext är viktigt

Varje LLM har ett ändligt **kontextfönster** — det maximala antalet tokens den kan bearbeta i en enda förfrågan. När en flerspårig konversation fortskrider:

- **Antalet tokens ökar linjärt** med varje användarmeddelande och assistentens svar.
- **Prompt-tokens dominerar kostnaden** eftersom hela historiken skickas om varje varv.
- Så småningom **överskrider konversationen kontextfönstret** och modellen antingen trunkerar eller ger fel.

### Strategier för att hantera kontext

| Strategi | Hur det fungerar | Kompromiss |
|---|---|---|
| **Trunkering** | Släpper de äldsta meddelandena | Förlorar tidig kontext |
| **Sammanfattning** | Komprimerar äldre meddelanden till en sammanfattning | Några detaljer går förlorade, men nyckelpunkter behålls |
| **Anteckningsblock / Externt minne** | Sparar viktiga fakta utanför konversationen | Kräver verktygsanrop, men överlever alla reduktioner |

I denna notebook kombinerar vi **sammanfattning** med ett **anteckningsblocksverktyg** så att agenten kan bibehålla kontinuitet även när konversationshistoriken kondenseras.


## Skapa en kontextmedveten agent


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulera en lång konversation

Låt oss gå igenom en konversation med flera turer för att se hur kontexten ackumuleras. Agenten bör behålla viktiga detaljer (preferenser, budget, resdatum) över turerna och visa kontinuitet.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Lägg märke till hur agenten behåller kontext från tidigare turer — den vet om Japan, sushi, tempel, fotografering, budgeten på 3000 dollar, resor ensam och tidsramen i april. I en kort konversation fungerar detta bra, men när konversationen växer blir hela historiken dyr att skicka om.

Låt oss fortsätta konversationen med fler turer för att se hur kontexten ackumuleras:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Mönster för sammanfattning av sammanhang

När konversationen växer kan vi använda ett **sammanfattningsverktyg** för att kondensera ackumulerat sammanhang till ett kompakt format. Agenten anropar detta verktyg för att registrera viktiga preferenser så att även om äldre meddelanden tas bort, bevaras den viktiga informationen.

Detta mönster är byggstenen för mer sofistikerad historikreducering:
1. Agenten identifierar viktiga fakta från konversationen
2. Den anropar sammanfattningsverktyget för att spara dem
3. Äldre meddelanden kan tryggt tas bort eftersom sammanfattningen fångar det som är viktigt

Nedan definierar vi ett `summarize_preferences` verktyg som agenten kan anropa för att registrera en kompakt sammanfattning av vad den har lärt sig.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Sammanfattning

I denna lektion lärde du dig hur man hanterar kontext i långvariga agentkonversationer med Microsoft Agent Framework:

### Viktiga begrepp
- **Kontextfönster är begränsade** — varje token i konversationshistoriken kostar pengar och räknas mot gränsen.
- **Sammanfattningsverktyg** låter agenten kondensera ackumulerad kontext till kompakta sammanfattningar, vilket minskar tokenanvändningen samtidigt som viktig information bevaras.
- **Agentens anteckningsblock** erbjuder ett bestående externt minne som överlever alla konversationsminskningar.

### Vad du byggde
- En **konstextkännande agent** som upprätthåller kontinuitet över flera turer i samtalet
- Ett **sammanfattningsverktyg** (`summarize_preferences`) som registrerar viktiga användardetaljer i ett kompakt format
- En **flerturns-konversation** som demonstrerar kontextbehållning och hantering av förändringar

### Verkliga tillämpningar
- **Kundtjänstbotar**: Kom ihåg preferenser under långa supportsessioner
- **Personliga assistenter**: Följ pågående projekt utan att behöva förklara kontext igen
- **Utbildningshandledare**: Behåll elevens framsteg över många interaktioner

### Nästa steg
- Implementera ett fullständigt anteckningsblockverktyg med filbaserad ihållande lagring
- Lägg till automatisk historikavkortning efter sammanfattning
- Kombinera med vektordatabaser för semantisk minnessökning
- Bygg agenter som kan återuppta konversationer flera dagar senare med full kontext


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
